# PG-Stack · Step 2 — Image Base Learners (GPU)

Two image base learners, trained on the **same patient-grouped folds** as Step 1:

| Base learner | What | Why |
|---|---|---|
| **`emb_lr`** | Frozen **DINOv2 ViT-S/14** embeddings → logistic regression | Foundation-model view; embeddings are also reused as **graph node features + kNN edges** in Step 4 |
| **`cnn`** | **EfficientNet-B0** fine-tuned per fold, fresh negative sampling each epoch | Task-specific image view |

**Kaggle settings (right panel):**
- *Add Input* → Competitions → `isic-2024-challenge`
- *Add Input* → **Your Work / Notebooks** → your **Step 1 notebook** (its saved output, which has `folds.csv`)
- Accelerator → **GPU T4 x2** or **P100**
- Internet → **ON** (to download pretrained weights; needs a phone-verified account)

Expected runtime ≈ 60–90 min. Use **Save Version → Save & Run All** so it runs in the background.

In [ ]:
import os, io, gc, glob, time, json, warnings
import numpy as np, pandas as pd, h5py
import torch, torch.nn as nn, timm
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import roc_curve, auc, roc_auc_score
warnings.filterwarnings("ignore")

# ---------------- Config ----------------
COMP_DIR = "/kaggle/input/isic-2024-challenge"
OUT_DIR  = "/kaggle/working"
_hits = glob.glob("/kaggle/input/**/folds.csv", recursive=True)
assert _hits, "Add your Step-1 notebook output as an input (needs folds.csv)"
STEP1_DIR = os.path.dirname(_hits[0])

SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
PRETRAINED = True

# Foundation-model embeddings
FM_NAME, FM_IMG, FM_BS = "vit_small_patch14_dinov2.lvd142m", 224, 256
PCA_DIM = 32

# Fine-tuned CNN
CNN_NAME, CNN_IMG, CNN_BS = "efficientnet_b0.ra_in1k", 160, 64
EPOCHS, NEG_PER_POS, LR = 8, 20, 3e-4
MAX_FOLDS = 5          # set lower to debug quickly

torch.manual_seed(SEED); np.random.seed(SEED)
print("device:", DEVICE, "| step1 outputs:", STEP1_DIR)
print("GPUs:", torch.cuda.device_count())

## 1. Folds + metric (identical to Step 1)

In [ ]:
folds = pd.read_csv(f"{STEP1_DIR}/folds.csv")
N_FOLDS = folds.fold.nunique()
print(folds.shape, "folds:", N_FOLDS, "pos:", folds.target.sum())

def pauc_80(y_true, y_pred, min_tpr=0.80):
    v_gt = np.abs(np.asarray(y_true) - 1); v_pred = -1.0 * np.asarray(y_pred)
    max_fpr = abs(1 - min_tpr)
    fpr, tpr, _ = roc_curve(v_gt, v_pred)
    stop = np.searchsorted(fpr, max_fpr, "right")
    tpr = np.append(tpr[:stop], np.interp(max_fpr, [fpr[stop-1], fpr[stop]], [tpr[stop-1], tpr[stop]]))
    fpr = np.append(fpr[:stop], max_fpr)
    return auc(fpr, tpr)

def report(name, y, p):
    neg = np.sort(p[y == 0]); thr = neg[int(0.95 * len(neg))]
    return dict(model=name, pauc80=pauc_80(y, p), auroc=roc_auc_score(y, p),
                **{"sens@spec95": float((p[y == 1] > thr).mean())})

## 2. HDF5 image dataset (opened lazily per worker)

In [ ]:
H5_PATH = f"{COMP_DIR}/train-image.hdf5"

class H5Dataset(Dataset):
    def __init__(self, ids, labels=None, tfm=None):
        self.ids, self.labels, self.tfm, self.h5 = list(ids), labels, tfm, None
    def __len__(self): return len(self.ids)
    def __getitem__(self, i):
        if self.h5 is None:
            self.h5 = h5py.File(H5_PATH, "r")
        img = Image.open(io.BytesIO(self.h5[self.ids[i]][()])).convert("RGB")
        x = self.tfm(img)
        y = -1 if self.labels is None else self.labels[i]
        return x, torch.tensor(y, dtype=torch.float32)

def loader(ids, labels, tfm, bs, shuffle=False):
    return DataLoader(H5Dataset(ids, labels, tfm), batch_size=bs, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"),
                      persistent_workers=False, drop_last=False)

with h5py.File(H5_PATH, "r") as f:
    print("images in hdf5:", len(f.keys()))

## 3. Base learner A — frozen DINOv2 embeddings for all ~401k lesions
Saved as `emb_dinov2_f16.npy` (row order = `folds.csv`). Reused in Step 4 for graph edges.

In [ ]:
fm = timm.create_model(FM_NAME, pretrained=PRETRAINED, num_classes=0, img_size=FM_IMG).to(DEVICE).eval()
cfg = timm.data.resolve_data_config({}, model=fm)
MEAN, STD = cfg["mean"], cfg["std"]
fm_tfm = T.Compose([T.Resize((FM_IMG, FM_IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])
if torch.cuda.device_count() > 1: fm = nn.DataParallel(fm)

emb_path = f"{OUT_DIR}/emb_dinov2_f16.npy"
t0, chunks = time.time(), []
with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
    for i, (x, _) in enumerate(loader(folds.isic_id, None, fm_tfm, FM_BS)):
        chunks.append(fm(x.to(DEVICE, non_blocking=True)).float().cpu().numpy().astype(np.float16))
        if i % 200 == 0: print(f"  batch {i} | {time.time()-t0:.0f}s")
EMB = np.concatenate(chunks); del chunks, fm; gc.collect(); torch.cuda.empty_cache()
np.save(emb_path, EMB)
print("embeddings:", EMB.shape, f"in {time.time()-t0:.0f}s")

In [ ]:
# Compact PCA version -> graph node features (unsupervised, no labels used)
rng = np.random.default_rng(SEED)
fit_idx = rng.choice(len(EMB), size=min(50000, len(EMB)), replace=False)
pca = PCA(n_components=PCA_DIM, random_state=SEED).fit(EMB[fit_idx].astype(np.float32))
EMB_PCA = pca.transform(EMB.astype(np.float32)).astype(np.float32)
np.save(f"{OUT_DIR}/emb_pca{PCA_DIM}.npy", EMB_PCA)
print("PCA explained variance:", round(pca.explained_variance_ratio_.sum(), 3))

In [ ]:
# Logistic-regression head on frozen embeddings (OOF)
y = folds.target.values
oof_emb = np.zeros(len(folds))
for f in range(N_FOLDS):
    tr = np.where(folds.fold != f)[0]; va = np.where(folds.fold == f)[0]
    tr_pos = tr[y[tr] == 1]
    tr_neg = rng.choice(tr[y[tr] == 0], size=min(len(tr_pos) * 50, (y[tr] == 0).sum()), replace=False)
    sub = np.r_[tr_pos, tr_neg]
    sc = StandardScaler().fit(EMB[sub].astype(np.float32))
    clf = LogisticRegression(C=0.05, max_iter=3000, class_weight="balanced")
    clf.fit(sc.transform(EMB[sub].astype(np.float32)), y[sub])
    oof_emb[va] = clf.predict_proba(sc.transform(EMB[va].astype(np.float32)))[:, 1]
    print(f"  [emb_lr] fold {f}: pAUC={pauc_80(y[va], oof_emb[va]):.4f}")
print("emb_lr overall pAUC:", round(pauc_80(y, oof_emb), 4))
del EMB, EMB_PCA; gc.collect()   # free RAM before CNN training (embeddings are saved to disk)

## 4. Base learner B — EfficientNet-B0 fine-tuned per fold
Each epoch: all training positives + a **fresh random set of negatives** (20 per positive). Predict the held-out fold with flip-TTA.

In [ ]:
train_tfm = T.Compose([
    T.RandomResizedCrop(CNN_IMG, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(30),
    T.ColorJitter(0.2, 0.2, 0.2, 0.02),
    T.ToTensor(), T.Normalize(MEAN, STD)])
valid_tfm = T.Compose([T.Resize((CNN_IMG, CNN_IMG)), T.ToTensor(), T.Normalize(MEAN, STD)])

def make_cnn():
    m = timm.create_model(CNN_NAME, pretrained=PRETRAINED, num_classes=1, drop_rate=0.3)
    m = m.to(DEVICE)
    return nn.DataParallel(m) if torch.cuda.device_count() > 1 else m

@torch.no_grad()
def predict(model, ids):
    model.eval(); out = []
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
        for x, _ in loader(ids, None, valid_tfm, CNN_BS * 4):
            x = x.to(DEVICE, non_blocking=True)
            p = (torch.sigmoid(model(x)) + torch.sigmoid(model(torch.flip(x, dims=[3])))) / 2
            out.append(p.float().squeeze(1).cpu().numpy())
    return np.concatenate(out)

oof_cnn = np.full(len(folds), np.nan)
ids_all = folds.isic_id.values
for f in range(min(N_FOLDS, MAX_FOLDS)):
    t0 = time.time()
    tr = np.where(folds.fold != f)[0]; va = np.where(folds.fold == f)[0]
    tr_pos, tr_neg = tr[y[tr] == 1], tr[y[tr] == 0]
    model = make_cnn()
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    steps = EPOCHS * int(np.ceil(len(tr_pos) * (1 + NEG_PER_POS) / CNN_BS))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(DEVICE == "cuda"))
    lossf = nn.BCEWithLogitsLoss()
    for ep in range(EPOCHS):
        neg = rng.choice(tr_neg, size=min(len(tr_pos) * NEG_PER_POS, len(tr_neg)), replace=False)
        idx = np.r_[tr_pos, neg]
        model.train(); tot = 0
        for x, yy in loader(ids_all[idx], y[idx], train_tfm, CNN_BS, shuffle=True):
            x, yy = x.to(DEVICE, non_blocking=True), yy.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(DEVICE == "cuda")):
                loss = lossf(model(x).squeeze(1), yy)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x)
        print(f"  fold {f} ep {ep}: loss={tot/len(idx):.4f}")
    oof_cnn[va] = predict(model, ids_all[va])
    torch.save((model.module if hasattr(model, "module") else model).state_dict(), f"{OUT_DIR}/cnn_fold{f}.pt")
    print(f"[cnn] fold {f}: pAUC={pauc_80(y[va], oof_cnn[va]):.4f}  ({(time.time()-t0)/60:.1f} min)")
    del model, opt; gc.collect(); torch.cuda.empty_cache()

## 5. Results + complementarity check

In [ ]:
tab = pd.read_csv(f"{STEP1_DIR}/oof_tabular.csv")
assert (tab.isic_id.values == folds.isic_id.values).all()
oof = tab.copy()
oof["oof_emb_lr"] = oof_emb
oof["oof_cnn"] = oof_cnn
done = ~np.isnan(oof_cnn)

def rank(a): return pd.Series(a).rank(pct=True).values
oof["avg_tab_img"] = (rank(oof.oof_tab_lesion) + rank(oof.oof_emb_lr) + rank(np.nan_to_num(oof.oof_cnn))) / 3

rows = [report(c.replace("oof_", ""), y[done], oof[c].values[done])
        for c in ["oof_tab_lesion", "oof_tab_pnorm", "oof_emb_lr", "oof_cnn", "avg_tab_img"]]
res = pd.DataFrame(rows).set_index("model").round(4)
print(res)
print("\nSpearman correlation between base learners (low = complementary):")
print(oof.loc[done, ["oof_tab_lesion", "oof_emb_lr", "oof_cnn"]].corr("spearman").round(3))

## 6. Save outputs for Steps 3–4

In [ ]:
oof.to_csv(f"{OUT_DIR}/oof_all.csv", index=False)
res.to_csv(f"{OUT_DIR}/results_step2.csv")
print(sorted(os.listdir(OUT_DIR)))